In [1]:
from datetime import datetime
from typing import Iterator, Any
from bs4 import BeautifulSoup
import requests
from requests import Response
from urllib.parse import urlencode, urljoin, urlparse, parse_qs

from src.models import ProductDB, QueryDB

In [86]:
def convert_date(date_raw: str) -> datetime:
    date = date_raw[date_raw.find("[") + 1:date_raw.find("]")]
    day, month, year = map(int, date.split('.'))
    date_formatted = datetime(day=day, month=month, year=year)
    return date_formatted

def extract_product(product_raw) -> ProductDB:
    header_tmp = product_raw.find(class_="inzeratynadpis")
    name_div = header_tmp.find(class_='nadpis')
    date_raw = header_tmp.find(class_='velikost10').text

    name = name_div.text
    name = ' '.join(name.split())
    url = name_div.select_one('a[href]').get('href')
    parsed_url_path = urlparse(url).path.split("/")
    bazos_id = parsed_url_path[2]
    # link = urljoin(BAZOS_BASE_URL, link)
    date = convert_date(date_raw)
    ignore_date = 'TOP' in date_raw
    description = header_tmp.find(class_='popis').text
    price = product_raw.find(class_='inzeratycena').text.strip()

    location_tmp = product_raw.find(class_='inzeratylok')
    location_tmp.br.replace_with(", ")
    location = location_tmp.text
    
    view_count_tmp = product_raw.find(class_='inzeratyview')
    view_count = int(view_count_tmp.text.split(' ')[0])
    print(f"{view_count=}")
    
    actions = product_elements[0].find(class_='inzeratyakce')
    product_id = int(parse_qs(urlparse(actions.find('a', string='Označit špatný inzerát').attrs['href']).query)['idad'][0])
    print(f"{product_id=}")
    
    return ProductDB(bazos_id=bazos_id, name=name, price=price, url=url, location=location, publish_date=date,
                     ignore_date=ignore_date, description=description)

In [87]:
response = requests.get("https://pc.bazos.cz/notebook/")

In [88]:
soup = BeautifulSoup(response.text, 'html.parser')
product_elements = soup.find_all(class_='inzeraty')

In [89]:
product = extract_product(product_elements[0])
product.__dict__

view_count=166
product_id=154512180


{'_sa_instance_state': <sqlalchemy.orm.state.InstanceState at 0x11207d900>,
 'bazos_id': '154512180',
 'name': '2K 16" LENOVO THINKBOOK 16P_RYZEN 9 5900HX_32GB_RTX 3060 6GB',
 'price': '27 500 Kč',
 'url': '/inzerat/154512180/2k-16-lenovo-thinkbook-16pryzen-9-5900hx32gbrtx-3060-6gb.php',
 'location': 'Děčín, 405 02',
 'publish_date': datetime.datetime(2022, 7, 2, 0, 0),
 'ignore_date': True,
 'description': '---JEN VOLAT---***PROFESIONÁLNÍ CELOKOVOVÝ NOTEBOOK***\r\nVEŠKERÉ FOTOGRAFIE JSOU REÁLNÉ - KUPUJETE CO VIDÍTE. KLIKNUTÍM NA JMÉNO KOUKNĚTE NA DALŠÍ MOJE NABÍDKY.\r\nChtěl bych zde nabídnout vysoce výkonný notebook pro náročné programy a hry\r\n\r\nLenovo Thi ...'}

In [50]:
actions = product_elements[0].find(class_='inzeratyakce')

In [80]:
int(parse_qs(urlparse(actions.find('a', string='Označit špatný inzerát').attrs['href']).query)['idad'][0])

154512180

In [96]:
publisher_info = parse_qs(urlparse(actions.find('a', string='Ohodnotit uživatele').attrs['href']).query)
publisher_info

{'idmail': ['1090722'], 'idphone': ['12956'], 'jmeno': ['PC LEVNĚ']}

In [97]:
int(publisher_info['idmail'][0])

1090722

In [98]:
int(publisher_info['idphone'][0])

12956

In [99]:
publisher_info['jmeno'][0]

'PC LEVNĚ'

In [4]:
response = requests.get("https://mobil.bazos.cz/inzerat/154577410/ipad-5-2017-128gb-space-grey-pekny-stav-12-mesicu-zaruka.php")

In [6]:
soup = BeautifulSoup(response.text, 'html.parser')

In [11]:
soup

<!DOCTYPE html>

<html lang="cs">
<head>
<title>iPad 5 (2017) 128GB space grey, pěkný stav, 12 měsíců záruka - Praha 2 | Bazoš.cz</title>
<meta content="text/html; charset=utf-8" http-equiv="Content-Type"/>
<meta content="Inzerát č. 154577410: iPad 5 (2017) 128GB space grey, pěkný stav, 12 měsíců záruka, Cena:   5 900 Kč, Lokalita: Praha 2" name="description"/><meta content="https://www.bazos.cz/img/1/410/154577410.jpg" property="og:image"/><link href="https://mobil.bazos.cz/inzerat/154577410/ipad-5-2017-128gb-space-grey-pekny-stav-12-mesicu-zaruka.php" rel="canonical"/><meta content="1055875657" property="fb:admins"/>
<meta content="max-image-preview:large" name="ROBOTS"/>
<link as="image" href="https://www.bazos.cz/img/1/410/154577410.jpg" rel="preload"/><link href="https://www.bazos.cz/bazos60s.css" rel="stylesheet" type="text/css"/><link as="image" href="https://www.bazos.cz/obrazky/bazos.svg" rel="preload"/>
<link href="https://www.bazos.cz/bazosprint.css" media="print" rel="style

In [10]:
print(soup.find(class_="popisdetail").text)

iPad 5 128GB šedý

- vše funkční se zárukou 12 měsíců
- do 30 dní možnost vyměnit za jiný
- odhlášený od iCloudu
- v ceně kabel a nabíječka

Vyzkoušet a zakoupit můžete na adrese Korunní 47, Praha 2. Nebo je možnost odeslání na dobírku.

Máme na prodej i mnoho jiných iPhonů.
